# Day 3 — 배운 문법으로 작은 유전자·임상 표 다루기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-1/day-03.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-1/day-03.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 유전자 표에서 log2 fold change 계산하고 라벨 붙이기

In [ ]:
import math

# 가상 데이터: 유전자 5개의 평균 발현량 (대조군 vs 비행군)
genes = [
    {"symbol": "Gfap",  "pathway": "신경염증",     "control": 10.0, "flight": 15.0},
    {"symbol": "Sod2",  "pathway": "산화스트레스", "control": 8.0,  "flight": 8.4},
    {"symbol": "Mbp",   "pathway": "수초",         "control": 20.0, "flight": 12.0},
    {"symbol": "Casp3", "pathway": "apoptosis",    "control": 5.0,  "flight": 7.5},
    {"symbol": "Syp",   "pathway": "시냅스",       "control": 16.0, "flight": 15.0},
]

def log2_fold_change(control, treated):
    """처리군/대조군 비율에 log2를 씌운 값을 돌려준다."""
    return math.log2(treated / control)

def label_direction(lfc, cutoff=0.5):
    """log2FC가 cutoff보다 크면 '증가', -cutoff보다 작으면 '감소', 아니면 '변화 작음'."""
    if lfc > cutoff:
        return "증가"
    elif lfc < -cutoff:
        return "감소"
    else:
        return "변화 작음"

for gene in genes:
    lfc = log2_fold_change(gene["control"], gene["flight"])
    print(f'{gene["symbol"]:<6} log2FC={lfc:+.2f}  {label_direction(lfc)}  ({gene["pathway"]})')

### 예시 2 — 임상 표에서 군별 인원과 평균 나이 구하기 (결측 포함)

In [ ]:
# 가상 데이터: 참가자 6명의 임상 정보 (None = 기록 없음)
participants = [
    {"id": "P001", "group": "PD",      "age": 67,   "updrs": 28},
    {"id": "P002", "group": "Control", "age": 64,   "updrs": 3},
    {"id": "P003", "group": "PD",      "age": 72,   "updrs": None},
    {"id": "P004", "group": "Control", "age": 59,   "updrs": 1},
    {"id": "P005", "group": "PD",      "age": None, "updrs": 35},
    {"id": "P006", "group": "PD",      "age": 58,   "updrs": 19},
]

def mean_ignoring_missing(values):
    """None을 빼고 평균을 계산한다. 남은 값이 없으면 None을 돌려준다."""
    present = [v for v in values if v is not None]
    if len(present) == 0:
        return None
    return sum(present) / len(present)

# 1) 군별 인원 세기
counts = {}
for p in participants:
    group = p["group"]
    counts[group] = counts.get(group, 0) + 1
print("군별 인원:", counts)

# 2) 군별 평균 나이와 결측 개수
for group in counts:
    ages = [p["age"] for p in participants if p["group"] == group]
    n_missing = ages.count(None)
    print(f"{group}: 평균 나이 = {mean_ignoring_missing(ages):.1f}세 (결측 {n_missing}명)")

# 3) 값이 빠진 참가자 찾기
for p in participants:
    missing = [key for key, value in p.items() if value is None]
    if missing:
        print(p["id"], "→ 빠진 항목:", missing)

### 예시 3 — 없는 키와 대소문자 함정

In [ ]:
gene_to_pathway = {
    "Gfap": "신경염증",
    "Sod2": "산화스트레스",
    "Mbp": "수초",
}

for symbol in ["Gfap", "Mbp", "Snca"]:
    pathway = gene_to_pathway.get(symbol, "미분류")
    print(symbol, "→", pathway)

# 대소문자가 다르면 다른 키다
print("GFAP" in gene_to_pathway)
print("GFAP".capitalize() in gene_to_pathway)

## 직접 해보기

1. **노트북을 만듭니다.** 터미널에서 프로젝트 폴더로 가서 환경을 켜고 JupyterLab을 엽니다.

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# cd ~/Documents/neurodegeneration-bioinformatics
# conda activate neurobio
# jupyter lab

5. **변형 2 — 필터 함수 만들기.** 아래 빈칸을 채워, 라벨이 "변화 작음"이 **아닌** 유전자의 이름만 리스트로 돌려주는 함수를 완성하세요.

In [ ]:
def changed_genes(gene_rows, cutoff=0.5):
    result = []
    for gene in gene_rows:
        lfc = log2_fold_change(gene["control"], gene["flight"])
        if label_direction(lfc, cutoff) != "변화 작음":
            result.append(____)
    return result

print(changed_genes(genes))

6. **변형 3 — 샘플 단위 표.** 지금까지는 "군 평균"이 이미 계산된 표였습니다. 실제 데이터는 샘플 하나하나의 값으로 옵니다. 아래를 실행하고, 코드가 한 줄씩 무엇을 하는지 주석(`#`)을 달아 보세요.

In [ ]:
samples = [
    {"sample": "S1", "group": "control", "Gfap": 9.8,  "Sod2": 8.1},
    {"sample": "S2", "group": "control", "Gfap": 10.4, "Sod2": 7.9},
    {"sample": "S3", "group": "flight",  "Gfap": 13.5, "Sod2": 8.6},
    {"sample": "S4", "group": "flight",  "Gfap": None, "Sod2": 8.2},
]

def group_mean(rows, group, gene):
    values = [r[gene] for r in rows if r["group"] == group and r[gene] is not None]
    return sum(values) / len(values)

for gene in ["Gfap", "Sod2"]:
    c = group_mean(samples, "control", gene)
    f = group_mean(samples, "flight", gene)
    print(f"{gene}: control={c:.2f}, flight={f:.2f}, 차이={f - c:+.2f}")

7. **저장, 일지, commit.** 노트북을 저장(`Cmd + S`)한 뒤 터미널에서:

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# git add notebooks/01_python_basics.ipynb learning-log.md
# git commit -m "Day 3: 리스트·딕셔너리·함수로 유전자/임상 표 처리"